# Assignment 10 — NLP Using Python-II
**Course:** UCS420 Cognitive Computing  
**Topics:** Feature extraction from text, sentiment analysis and text generation

This notebook answers all six questions from Lab Assignment 10. Run the cells from top to bottom.

## 0. Install and import libraries

In [ ]:
%pip -q install nltk scikit-learn textblob wordcloud matplotlib tensorflow

In [ ]:
import re, warnings
import nltk
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from wordcloud import WordCloud
warnings.filterwarnings("ignore")
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)
STOP_WORDS = set(stopwords.words("english"))
print("Libraries loaded.")

# Q1. Text preprocessing and word frequency

**Favorite topic: Cricket**

Cricket interests me because it combines individual skill with team strategy. A batter must judge the ball quickly, choose the right shot, and remain calm under pressure. Bowlers use speed, swing, and variation to challenge the batter. Fielders support the team through alert movement and accurate throws. Modern cricket also uses data and technology to analyze performance and improve decisions. This balance of skill, teamwork, and analysis makes cricket fascinating to me.

In [ ]:
paragraph = (
    "Cricket interests me because it combines individual skill with team strategy. "
    "A batter must judge the ball quickly, choose the right shot, and remain calm under pressure. "
    "Bowlers use speed, swing, and variation to challenge the batter. "
    "Fielders support the team through alert movement and accurate throws. "
    "Modern cricket also uses data and technology to analyze performance and improve decisions. "
    "This balance of skill, teamwork, and analysis makes cricket fascinating to me."
)
# 1. Lowercase and remove punctuation using re
clean_text = re.sub(r"[^\w\s]", "", paragraph.lower())
print("Cleaned text:\n", clean_text)

# 2. Word and sentence tokenization
word_tokens = word_tokenize(paragraph)
sentence_tokens = sent_tokenize(paragraph)
print("\nWord tokens:", word_tokens)
print("\nSentence tokens:")
for i, sentence in enumerate(sentence_tokens, 1):
    print(i, sentence)

# 3. Compare split() and word_tokenize()
split_tokens = paragraph.split()
print("\nsplit() tokens:", split_tokens)
print("\nword_tokenize() tokens:", word_tokens)
print("\nComparison: split() separates on whitespace and leaves punctuation attached. "
      "word_tokenize() handles punctuation as separate tokens and applies language-aware tokenization.")

# 4. Remove stopwords
filtered_tokens = [w.lower() for w in word_tokens
                   if w.isalpha() and w.lower() not in STOP_WORDS]
print("\nAfter stopword removal:", filtered_tokens)

# 5. Word frequency distribution
freq_dist = FreqDist(filtered_tokens)
print("\nFrequency distribution:", freq_dist)
plt.figure(figsize=(10, 4))
freq_dist.plot(15, title="Word Frequency (stopwords excluded)")
plt.show()

# Q2. Feature extraction, stemming and lemmatization

The same paragraph is reused. Stemming removes word endings using rules; lemmatization attempts to return a dictionary base form. WordNetLemmatizer uses noun as its default part of speech.

In [ ]:
# 1. Extract alphabetic words using re.findall()
alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", paragraph.lower())
print("Alphabetic words:", alphabetic_words)

# 2. Remove stopwords
q2_words = [w for w in alphabetic_words if w not in STOP_WORDS]
print("\nAfter stopword removal:", q2_words)

# 3. Porter stemming
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(w) for w in q2_words]

# 4. WordNet lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in q2_words]
print("\nStemmed:", stemmed_words)
print("\nLemmatized:", lemmatized_words)

print("\nOriginal | Stemmed | Lemmatized")
for original, stem, lemma in zip(q2_words, stemmed_words, lemmatized_words):
    print(f"{original:15} | {stem:15} | {lemma}")

print("\nExplanation:")
print("- Stemming is fast and useful in search/indexing when related word forms should be grouped.")
print("- Lemmatization is preferable when readable, linguistically meaningful base words are needed.")
print("- Stemming may produce non-dictionary forms; lemmatization is more language-aware.")

# Q3. Bag of Words and TF-IDF

Three original short texts are used. CountVectorizer records word counts. TfidfVectorizer assigns higher weight to terms that are important in one text but less common across the collection.

In [ ]:
texts = [
    "Artificial intelligence helps computers learn patterns and make useful predictions.",
    "Electric vehicles reduce fuel use and can make city transport cleaner.",
    "Cricket teams analyze player performance using statistics and video technology."
]
# 1. Bag of Words
count_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = count_vectorizer.fit_transform(texts)
print("Vocabulary:", list(count_vectorizer.get_feature_names_out()))
print("\nBag of Words matrix:")
print(bow_matrix.toarray())

# 2. TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
features = tfidf_vectorizer.get_feature_names_out()
print("\nTF-IDF matrix (rounded):")
print(tfidf_matrix.toarray().round(3))

# 3. Top 3 keywords from each text
print("\nTop 3 TF-IDF keywords per text:")
for i, row in enumerate(tfidf_matrix.toarray()):
    indices = row.argsort()[::-1][:3]
    print(f"Text {i+1}:", [(features[j], round(row[j], 3)) for j in indices])

print("\nInterpretation: High TF-IDF terms are relatively important to a particular text. "
      "The top terms summarize its main subject; results depend on the chosen document collection.")

# Q4. Similarity between two technology descriptions

The two descriptions are about Artificial Intelligence and Blockchain. Jaccard compares unique-word overlap, while cosine similarity compares TF-IDF weighted document vectors.

In [ ]:
technology_1 = (
    "Artificial intelligence enables computer systems to learn patterns from data. "
    "It supports prediction, automation, language processing, and image recognition. "
    "AI applications are used in healthcare, education, transport, and business. "
    "Its performance depends on data quality, algorithms, and computing resources."
)
technology_2 = (
    "Blockchain is a distributed digital ledger shared across a network. "
    "It records transactions in linked blocks that are difficult to alter. "
    "Blockchain supports decentralized applications, digital assets, and traceability. "
    "Its operation depends on network consensus, cryptography, and participating nodes."
)

def preprocess(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

tokens_1, tokens_2 = preprocess(technology_1), preprocess(technology_2)
set_1, set_2 = set(tokens_1), set(tokens_2)
print("AI tokens:", tokens_1)
print("\nBlockchain tokens:", tokens_2)

# a. Jaccard similarity
jaccard = len(set_1 & set_2) / len(set_1 | set_2)
print("\nJaccard similarity:", round(jaccard, 4))

# b. TF-IDF cosine similarity
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([technology_1, technology_2])
cosine = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
print("Cosine similarity:", round(float(cosine), 4))

# c. Analysis
print("\nAnalysis: Jaccard treats each unique word equally and ignores repetition. "
      "TF-IDF cosine similarity weights terms and compares document-vector direction. "
      "For these descriptions, cosine can give more informative weighted vocabulary overlap, "
      "while Jaccard is simpler to interpret as set overlap.")

# Q5. Sentiment analysis and word cloud

TextBlob provides polarity (−1 to +1) and subjectivity (0 to 1). Reviews are classified as Positive, Negative or Neutral using a small polarity threshold. The word cloud uses all positive reviews.

In [ ]:
reviews = [
    "The wireless headphones have clear sound, comfortable ear cushions, and excellent battery life. I am very happy with this purchase.",
    "The phone case looks attractive, but the material feels weak and the buttons are difficult to press. I am disappointed.",
    "The delivery was on time and the packaging was fine. The product works as expected, although nothing feels exceptional."
]

def classify_sentiment(polarity):
    if polarity > 0.10:
        return "Positive"
    elif polarity < -0.10:
        return "Negative"
    return "Neutral"

results = []
for i, review in enumerate(reviews, 1):
    sentiment = TextBlob(review).sentiment
    label = classify_sentiment(sentiment.polarity)
    results.append({"review": review, "polarity": sentiment.polarity,
                    "subjectivity": sentiment.subjectivity, "label": label})
    print(f"Review {i}: {review}")
    print(f"Polarity: {sentiment.polarity:.3f}")
    print(f"Subjectivity: {sentiment.subjectivity:.3f}")
    print(f"Class: {label}\n")

positive_text = " ".join(r["review"] for r in results if r["label"] == "Positive")
if positive_text.strip():
    cloud = WordCloud(width=900, height=450, background_color="white").generate(positive_text)
    plt.figure(figsize=(12, 6))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud — Positive Reviews")
    plt.show()
else:
    print("No positive reviews available for word cloud.")

print("Interpretation: Positive polarity indicates favorable wording; negative polarity "
      "indicates unfavorable wording; values near zero are neutral. Subjectivity estimates "
      "how opinion-based the text is. Lexicon-based analysis may miss sarcasm and context.")

# Q6. Text generation using an LSTM

A paragraph of approximately 100 words is used as training data. Keras Tokenizer converts words into integer IDs. N-gram sequences are created and a small LSTM predicts the next word. Text is generated from a seed word.

**Limitation:** A short training paragraph is insufficient for high-quality language generation. The result demonstrates the method and may be repetitive or grammatically imperfect.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

training_text = (
    "Technology changes the way people learn and communicate. "
    "Students use digital tools to explore ideas and solve problems. "
    "Computers process information and help people make decisions. "
    "Artificial intelligence can recognize patterns in large collections of data. "
    "Good technology should be useful, reliable, and easy to understand. "
    "Learning new skills helps people adapt to a changing world. "
    "Careful thinking remains important when using automated systems. "
    "People can combine creativity with technology to build better solutions. "
    "Practice improves understanding and makes difficult tasks easier. "
    "The future will bring new tools and new opportunities for learning."
)
tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1
input_sequences = []

for sentence in training_text.split("."):
    sequence = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(sequence)):
        input_sequences.append(sequence[:i+1])

max_len = max(len(seq) for seq in input_sequences)
padded = pad_sequences(input_sequences, maxlen=max_len, padding="pre")
X = padded[:, :-1]
y = tf.keras.utils.to_categorical(padded[:, -1], num_classes=total_words)

print("Vocabulary size:", total_words)
print("Training sequences:", len(input_sequences))
print("X shape:", X.shape, "y shape:", y.shape)

In [ ]:
model = Sequential([
    Embedding(total_words, 32, input_length=max_len - 1),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()
history = model.fit(X, y, epochs=150, verbose=0)
print("Training complete.")

plt.figure(figsize=(7, 3))
plt.plot(history.history["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Training Loss")
plt.show()

In [ ]:
def generate_text(seed_text, next_words=24):
    output = seed_text.lower()
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([output])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding="pre")
        probabilities = model.predict(seq, verbose=0)[0]
        predicted_id = int(np.argmax(probabilities))
        if predicted_id == 0:
            break
        predicted_word = tokenizer.index_word.get(predicted_id, "")
        if not predicted_word:
            break
        output += " " + predicted_word
    return output

seed_word = "technology"
print("Seed:", seed_word)
print("\nGenerated text (approximately 2–3 lines):")
print(generate_text(seed_word, next_words=24))

## Final conclusion

This assignment demonstrates:
- **Q1:** cleaning, word/sentence tokenization, stopword removal and word frequency.
- **Q2:** alphabetic word extraction, stemming and lemmatization.
- **Q3:** Bag of Words and TF-IDF representations with keyword interpretation.
- **Q4:** Jaccard and TF-IDF cosine similarity.
- **Q5:** sentiment polarity, subjectivity, review classification and word cloud.
- **Q6:** Keras tokenization, n-gram sequences, LSTM training and seed-based text generation.

### References
- NLTK: https://www.nltk.org/
- Scikit-learn: https://scikit-learn.org/stable/modules/feature_extraction.html
- TextBlob: https://textblob.readthedocs.io/
- WordCloud: https://github.com/amueller/word_cloud
- TensorFlow/Keras: https://www.tensorflow.org/

**End of Assignment 10.**